In [1]:
"""
Exploration plots for gw-surrogate.
 
Generates figures showing what the training data looks like:
  1. Amplitude vs frequency for several (m1, m2) pairs
  2. Phase vs frequency for the same pairs
  3. Parameter space coverage (m1 vs m2 scatter)
  4. Waveform in time domain (inverse FFT) for intuition
 
Save these to figures/ for the README and interview.
Run: python exploration.py
"""

'\nExploration plots for gw-surrogate.\n \nGenerates figures showing what the training data looks like:\n  1. Amplitude vs frequency for several (m1, m2) pairs\n  2. Phase vs frequency for the same pairs\n  3. Parameter space coverage (m1 vs m2 scatter)\n  4. Waveform in time domain (inverse FFT) for intuition\n \nSave these to figures/ for the README and interview.\nRun: python exploration.py\n'

In [2]:
import numpy as np
import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
# add src to path so we can import the waveform function
import sys
sys.path.insert(0, "src")
from generate_data import compute_taylorf2
 
 
out = Path("figures")
out.mkdir(exist_ok=True)
 

In [4]:
# ── load training data ──────────────────────────────────────────
with h5py.File("data/train.h5", "r") as f:
    params = f["params"][:]
    log_amp = f["log_amplitude"][:]
    phase = f["phase"][:]
    freqs = f["f_array"][:]
 
m1_all = params[:, 0]
m2_all = params[:, 1]
 

In [5]:
# ── pick a handful of representative systems ────────────────────
examples = [
    (5.0,  5.0,  "equal low-mass"),
    (10.0, 10.0, "equal mid-mass"),
    (20.0, 20.0, "equal high-mass"),
    (15.0, 5.0,  "asymmetric q=1/3"),
    (20.0, 10.0, "asymmetric q=1/2"),
]
 
fig_amp, ax_amp = plt.subplots(figsize=(8, 5))
fig_pha, ax_pha = plt.subplots(figsize=(8, 5))
 
for m1, m2, label in examples:
    amp, phi = compute_taylorf2(m1, m2, freqs)
    tag = f"$m_1={m1:.0f},\\; m_2={m2:.0f}\\;M_\\odot$ ({label})"
 
    ax_amp.plot(freqs, np.log10(amp), label=tag)
    ax_pha.plot(freqs, phi, label=tag)
 
ax_amp.set_xlabel("Frequency [Hz]")
ax_amp.set_ylabel("$\\log_{10} |\\tilde{h}(f)|$")
ax_amp.set_title("TaylorF2 amplitude (Newtonian order)")
ax_amp.legend(fontsize=8)
ax_amp.grid(alpha=0.3)
fig_amp.tight_layout()
fig_amp.savefig(out / "amplitude_vs_freq.png", dpi=150)
print("saved amplitude_vs_freq.png")
 
ax_pha.set_xlabel("Frequency [Hz]")
ax_pha.set_ylabel("$\\Psi(f)$ [rad]")
ax_pha.set_title("TaylorF2 phase (1.5PN order)")
ax_pha.legend(fontsize=8)
ax_pha.grid(alpha=0.3)
fig_pha.tight_layout()
fig_pha.savefig(out / "phase_vs_freq.png", dpi=150)
print("saved phase_vs_freq.png")
 

saved amplitude_vs_freq.png
saved phase_vs_freq.png


In [6]:
# ── parameter space coverage ────────────────────────────────────
fig_par, ax_par = plt.subplots(figsize=(6, 5))
ax_par.scatter(m1_all, m2_all, s=1, alpha=0.15, c="steelblue")
ax_par.plot([5, 20], [5, 20], "k--", lw=0.8, label="$m_1 = m_2$")
ax_par.set_xlabel("$m_1$ [$M_\\odot$]")
ax_par.set_ylabel("$m_2$ [$M_\\odot$]")
ax_par.set_title("Training set parameter coverage (10k samples)")
ax_par.legend()
ax_par.set_aspect("equal")
ax_par.grid(alpha=0.3)
fig_par.tight_layout()
fig_par.savefig(out / "parameter_space.png", dpi=150)
print("saved parameter_space.png")
 
 

saved parameter_space.png


In [7]:
# ── distribution of phase and log-amplitude values ──────────────
fig_dist, (ax_d1, ax_d2) = plt.subplots(1, 2, figsize=(10, 4))
 
ax_d1.hist(log_amp.ravel(), bins=100, color="steelblue", alpha=0.7)
ax_d1.set_xlabel("$\\log_{10} |\\tilde{h}|$")
ax_d1.set_ylabel("count")
ax_d1.set_title("Log-amplitude distribution (all freq bins)")
 
ax_d2.hist(phase.ravel(), bins=100, color="coral", alpha=0.7)
ax_d2.set_xlabel("$\\Psi$ [rad]")
ax_d2.set_ylabel("count")
ax_d2.set_title("Phase distribution (all freq bins)")
 
fig_dist.tight_layout()
fig_dist.savefig(out / "data_distributions.png", dpi=150)
print("saved data_distributions.png")
 
 
plt.close("all")
print("\ndone — all figures in figures/")
 

saved data_distributions.png

done — all figures in figures/
